# Test realtime

In [ ]:
import os
import cv2
import numpy as np
import json
from pathlib import Path
from tqdm import tqdm

import mediapipe as mp
mp_holistic = mp.solutions.holistic
mp_face_mesh = mp.solutions.face_mesh

### Code trích xuất keypoint

In [ ]:
def _unique_indices(connections):
    idx = set()
    for a, b in connections:
        idx.add(a); idx.add(b)
    return idx

_lips = _unique_indices(mp_face_mesh.FACEMESH_LIPS)
_left_eye = _unique_indices(mp_face_mesh.FACEMESH_LEFT_EYE)
_left_eyebrow = _unique_indices(mp_face_mesh.FACEMESH_LEFT_EYEBROW)
_right_eye = _unique_indices(mp_face_mesh.FACEMESH_RIGHT_EYE)
_right_eyebrow = _unique_indices(mp_face_mesh.FACEMESH_RIGHT_EYEBROW)

FACE_SUBSET_IDX = sorted(_lips | _left_eye | _left_eyebrow | _right_eye | _right_eyebrow)
N_FACE = len(FACE_SUBSET_IDX)

# Vị trí (trong FACE_SUBSET_IDX) của 2 điểm khoé mắt ngoài, dùng để scale
FACE_SCALE_I = FACE_SUBSET_IDX.index(33)   # khoé mắt phải (ngoài)
FACE_SCALE_J = FACE_SUBSET_IDX.index(263)  # khoé mắt trái (ngoài)

print(f"Số điểm face subset: {N_FACE}")
print(f"Feature vector/frame: pose(132) + face({N_FACE*3}) + left_hand(63) + right_hand(63) "
      f"= {132 + N_FACE*3 + 63 + 63}")


def extract_frame_landmarks(image_rgb, holistic_model):
    results = holistic_model.process(image_rgb)

    if results.pose_landmarks:
        pose = np.array(
            [[lm.x, lm.y, lm.z, lm.visibility] for lm in results.pose_landmarks.landmark],
            dtype=np.float32,
        )
    else:
        pose = np.full((33, 4), np.nan, dtype=np.float32)

    if results.face_landmarks:
        all_face = results.face_landmarks.landmark
        face = np.array(
            [[all_face[i].x, all_face[i].y, all_face[i].z] for i in FACE_SUBSET_IDX],
            dtype=np.float32,
        )
    else:
        face = np.full((N_FACE, 3), np.nan, dtype=np.float32)

    if results.left_hand_landmarks:
        lh = np.array(
            [[lm.x, lm.y, lm.z] for lm in results.left_hand_landmarks.landmark],
            dtype=np.float32,
        )
    else:
        lh = np.full((21, 3), np.nan, dtype=np.float32)

    if results.right_hand_landmarks:
        rh = np.array(
            [[lm.x, lm.y, lm.z] for lm in results.right_hand_landmarks.landmark],
            dtype=np.float32,
        )
    else:
        rh = np.full((21, 3), np.nan, dtype=np.float32)

    return pose, face, lh, rh

Số điểm face subset: 92
Feature vector/frame: pose(132) + face(276) + left_hand(63) + right_hand(63) = 534


In [ ]:
# Hàm fill missing cho 1 feature của 1 frame
def fill_missing(raw):
    """
    raw: array (K, D) biểu diễn landmarks của 1 feature pose/face/lh/rh
    Trả về: array (K, D)
    """
    # Nếu feature này absent suốt video -> zero-fill
    if np.all(np.isnan(raw)):
        return np.zeros(raw.shape, dtype=np.float32)
    return raw.astype(np.float32)


# Hàm normalize cho 1 feature của 1 frame
def normalize(raw, origin, scale_idx_pair, coord_dims=3):
    """
    raw: array (K, D) của 1 nhóm tại 1 frame (đã qua fill_missing)
    origin: int (index trong K) HOẶC "centroid"
    scale_idx_pair: (i, j) - 2 index dùng tính khoảng cách làm scale
    coord_dims: số chiều đầu (x,y,z) cần normalize; chiều sau (vd visibility) giữ nguyên
    """
    seq = raw.copy()
    if np.all(seq == 0):
        return seq  # nhóm absent, giữ nguyên 0

    coords = seq[:, :coord_dims]  # (K, coord_dims)

    if origin == "centroid":
        origin_pt = coords.mean(axis=0, keepdims=True)  # (1, coord_dims)
    else:
        origin_pt = coords[origin:origin+1, :]

    centered = coords - origin_pt

    i, j = scale_idx_pair
    scale = np.linalg.norm(coords[j] - coords[i])
    scale = max(scale, 1e-6)

    seq[:, :coord_dims] = centered / scale
    return seq


def normalize_all(lh, rh):
    """
    lh, rh: mỗi cái là array (K, D) của 1 frame duy nhất (đã qua fill_missing)
    """
    lh_n = normalize(lh, origin=0, scale_idx_pair=(0, 9), coord_dims=3)
    rh_n = normalize(rh, origin=0, scale_idx_pair=(0, 9), coord_dims=3)
    return lh_n, rh_n

In [ ]:
def preprocess(lh_raw, rh_raw):  # (K, D)
    lh_rs = fill_missing(lh_raw)
    rh_rs = fill_missing(rh_raw)

    lh_n, rh_n = normalize_all(lh_rs, rh_rs)  # (K, D)

    return np.concatenate([lh_n.reshape(-1), rh_n.reshape(-1)])   # (126,)

### Chuẩn bị label và mô hình

In [ ]:
# Tải label
RES_DIR = '../results/keypoints_output_holistic'
with open(f'{RES_DIR}/label_classes.json', 'r', encoding='utf-8') as f:
    class_names = json.load(f)
num_classes = len(class_names)  # 30
print(f"Đã tải {num_classes} nhãn thành công.")

Đã tải 30 nhãn thành công.


In [ ]:
import tensorflow as tf
model_path = "../models/cnn1d_model_hand.h5" 

# 2. Tải mô hình vào bộ nhớ
model = tf.keras.models.load_model(model_path)
print("Tải mô hình thành công!")
model.input_shape

c:\ProgramData\miniconda3\envs\keras3_env\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Tải mô hình thành công!


(None, 30, 126)

### Code visualize

In [ ]:
from PIL import Image, ImageDraw, ImageFont

COLORS = [
    (0, 255, 0),      # Green
    (255, 255, 0),    # Cyan
    (255, 165, 0),    # Orange
    (255, 0, 255),    # Purple
    (128, 128, 128)   # Gray
]

def prob_viz(res, class_names, input_frame, top_k=5):
    output_frame = input_frame.copy()

    top_indices = np.argsort(res)[-top_k:][::-1]

    # OpenCV BGR -> PIL RGB
    output_pil = Image.fromarray(
        cv2.cvtColor(output_frame, cv2.COLOR_BGR2RGB)
    )

    draw = ImageDraw.Draw(output_pil)

    # Font tiếng Việt trên Windows
    font = ImageFont.truetype(
        "C:/Windows/Fonts/arial.ttf",
        20
    )

    for rank, idx in enumerate(top_indices):
        prob = res[idx]
        label = class_names[idx]

        y = 70 + rank * 35

        # Màu BGR -> RGB
        color = COLORS[rank][::-1]

        # Vẽ thanh xác suất
        draw.rectangle(
            [(0, y), (int(prob * 200), y + 25)],
            fill=color
        )

        # Vẽ chữ tiếng Việt
        draw.text(
            (205, y),
            f"{label}: {prob:.2f}",
            font=font,
            fill=(255, 255, 255)
        )

    # PIL RGB -> OpenCV BGR
    output_frame = cv2.cvtColor(
        np.array(output_pil),
        cv2.COLOR_RGB2BGR
    )

    return output_frame

In [ ]:
# Font hỗ trợ tiếng Việt trên Windows
font = ImageFont.truetype(
    "C:/Windows/Fonts/arial.ttf",
    30
)

def put_text_vietnamese(frame, text, position, font, color=(255, 255, 255)):
    # OpenCV BGR -> PIL RGB
    frame_pil = Image.fromarray(
        cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    )

    draw = ImageDraw.Draw(frame_pil)

    # PIL sử dụng RGB, OpenCV sử dụng BGR
    draw.text(
        position,
        text,
        font=font,
        fill=(color[2], color[1], color[0])
    )

    # PIL RGB -> OpenCV BGR
    return cv2.cvtColor(
        np.array(frame_pil),
        cv2.COLOR_RGB2BGR
    )

### Vòng lặp chính

In [ ]:
# 1. New detection variables
sequence = []
sentence = []
predictions = []
threshold = 0.5

cap = cv2.VideoCapture(0)
# Set mediapipe model
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():

        # Read feed
        ret, frame = cap.read()
  
        # # Make detections
        # image, results = mediapipe_detection(frame, holistic)
        # print(results)

        # # Draw landmarks
        # draw_styled_landmarks(image, results)

        # 2. Prediction logic
        pose_raw, face_raw, lh_raw, rh_raw = extract_frame_landmarks(frame, holistic)   # (K, D)
        keypoints = preprocess(lh_raw, rh_raw)
        sequence.append(keypoints)
        sequence = sequence[-30:]

        if len(sequence) == 30:
            res = model.predict(np.expand_dims(sequence, axis=0))[0]
            print(class_names[np.argmax(res)])
            predictions.append(np.argmax(res))

        #3. Viz logic
            if np.unique(predictions[-10:])[0]==np.argmax(res):
                if res[np.argmax(res)] > threshold:
                    if len(sentence) > 0:
                        if class_names[np.argmax(res)] != sentence[-1]:
                            sentence.append(class_names[np.argmax(res)])
                    else:
                        sentence.append(class_names[np.argmax(res)])

            if len(sentence) > 5:
                sentence = sentence[-5:]

            # Viz probabilities
            frame = prob_viz(res, class_names, frame)

        cv2.rectangle(frame, (0,0), (640, 40), (245, 117, 16), -1)
        # cv2.putText(frame, ' '.join(sentence), (3,30),
        #                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
        # Hiển thị tiếng Việt bằng Pillow
        frame = put_text_vietnamese(
            frame,
            ' '.join(sentence),
            (3, 3),
            font,
            (255, 255, 255)
        )
        cv2.imshow('OpenCV Feed', frame)    # Show to screen

        # Break gracefully
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break
    cap.release()
    cv2.destroyAllWindows()

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 504ms/step
Giúp
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
Giúp
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
Giúp
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
Giúp
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
Giúp
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
Giúp
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
Giúp
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
Giúp
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
Giúp
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
Giúp
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step
Giúp
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
Giúp
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
Giúp
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
Giúp
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step
Biết
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
Biết
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
Biết
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
Biết
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
Biết
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
Biết
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
Biết
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
Biết
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
Biết
1/1 ━━━